# AI Surveillance - Object Tracking & Restricted Zone Intrusion Demo

This notebook demonstrates the core AI engine of the surveillance platform: **YOLOv11 Object Tracking** combined with **Restricted Zone Intrusion Detection**.

### Google Colab Instructions:
1. Click the "Runtime" menu option and select **"Change runtime type"**. Make sure **T4 GPU** (or any GPU) is selected for optimal performance.
2. Run the cells sequentially to install dependencies, run inference, check for zone intrusions, and visualize results.

In [ ]:
# Install dependencies (ultralytics and opencv-python)
!pip install ultralytics opencv-python matplotlib numpy

In [ ]:
# Import libraries
import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
import os
%matplotlib inline

## 1. Initialize YOLOv11 Model

We load the YOLOv11 nano model (`yolo11n.pt`) which is pre-trained on the COCO dataset and supports tracking.

In [ ]:
print("Loading YOLO model...")
model = YOLO("yolo11n.pt")
print("YOLO model loaded successfully!")

## 2. Load a Test Image

To make YOLO detect a person, we will download a sample image containing a bus and a person from the official Ultralytics repo.

In [ ]:
import urllib.request
image_url = "https://raw.githubusercontent.com/ultralytics/ultralytics/main/ultralytics/assets/bus.jpg"
urllib.request.urlretrieve(image_url, "test_person.jpg")

# Load the real test image and resize it
frame = cv2.imread("test_person.jpg")
frame = cv2.resize(frame, (640, 480))
height, width = frame.shape[:2]

# Display the original test image
plt.figure(figsize=(8, 6))
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.title("Input Image")
plt.axis('off')
plt.show()

## 3. Define the Restricted Zone

We define a polygon representing the restricted zone on our camera view (normalized coordinates between 0 and 1).

In [ ]:
# Define restricted zone polygon vertices (normalized coordinates)
# This polygon covers a region in the center-left where the people are standing
normalized_polygon = [
    [0.05, 0.40],
    [0.60, 0.40],
    [0.65, 0.95],
    [0.02, 0.95]
]

# Scale to actual pixels
scaled_points = []
for pt in normalized_polygon:
    scaled_points.append([int(pt[0] * width), int(pt[1] * height)])
scaled_polygon = np.array(scaled_points, dtype=np.int32).reshape((-1, 1, 2))

print("Restricted zone polygon vertices (pixels):")
print(scaled_polygon.tolist())

## 4. Run Detection and Intrusion Check

We run YOLO tracking on the frame, check if any detected "person" is inside the polygon using a point-in-polygon test, and prepare prediction metadata.

In [ ]:
# Run YOLO tracking inference (conf=0.25, class 0 = person)
results = model.track(source=frame, persist=True, conf=0.25, classes=[0], imgsz=320, verbose=False)

predictions = []
zone_intruded = False
annotated_frame = frame.copy()

if results and len(results) > 0:
    result = results[0]
    boxes = result.boxes
    
    for i in range(len(boxes)):
        box = boxes[i]
        track_id = int(box.id[0].item()) if box.id is not None else 1
        class_id = int(box.cls[0].item())
        class_name = model.names[class_id]
        confidence = float(box.conf[0].item())
        
        xyxy = box.xyxy[0].cpu().numpy()
        x1, y1, x2, y2 = map(int, xyxy)
        
        # Check if center of detection box is inside the restricted zone
        center_x = (x1 + x2) // 2
        center_y = (y1 + y2) // 2
        point = (float(center_x), float(center_y))
        
        # cv2.pointPolygonTest checks if point is inside (> 0), on border (= 0), or outside (< 0)
        is_inside = cv2.pointPolygonTest(scaled_polygon, point, measureDist=False) >= 0
        if is_inside:
            zone_intruded = True
            
        predictions.append({
            "track_id": track_id,
            "class_name": class_name,
            "confidence": confidence,
            "box": [x1, y1, x2, y2],
            "intruded": is_inside
        })

print(f"Detected objects: {len(predictions)}")
print(f"Intrusion detected: {zone_intruded}")

## 5. Draw Overlays & Plot the Result

We overlay the restricted zone (red if intruded, cyan if clear) and the bounding boxes on the frame using OpenCV, then display it using Matplotlib.

In [ ]:
# Draw Restricted Zone overlay
overlay = annotated_frame.copy()
zone_color = (0, 0, 255) if zone_intruded else (255, 255, 0) # BGR: Red if intruded, Cyan/Yellow-green if clear
cv2.fillPoly(overlay, [scaled_polygon], zone_color)
cv2.addWeighted(overlay, 0.15, annotated_frame, 0.85, 0, annotated_frame)
cv2.polylines(annotated_frame, [scaled_polygon], isClosed=True, color=zone_color, thickness=2)

# Draw Detections
for pred in predictions:
    box = pred["box"]
    x1, y1, x2, y2 = box
    intruded = pred["intruded"]
    box_color = (0, 0, 255) if intruded else (0, 255, 0) # Red for intruder, Green for safe
    
    # Draw Bounding Box
    cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), box_color, 2)
    
    # Draw label
    label = f"{pred['class_name'].upper()} #{pred['track_id']} {int(pred['confidence']*100)}%"
    (text_w, text_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.4, 1)
    cv2.rectangle(annotated_frame, (x1, y1 - text_h - 6), (x1 + text_w + 10, y1), box_color, -1)
    cv2.putText(annotated_frame, label, (x1 + 5, y1 - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255) if intruded else (0, 0, 0), 1)
    
    # Draw center dot
    center_x = (x1 + x2) // 2
    center_y = (y1 + y2) // 2
    cv2.circle(annotated_frame, (center_x, center_y), 4, box_color, -1)

# Draw Warning Intrusion Header Banner
if zone_intruded:
    cv2.rectangle(annotated_frame, (0, 0), (width, 35), (0, 0, 255), -1)
    cv2.putText(annotated_frame, "!!! INTRUSION DETECTED IN RESTRICTED ZONE !!!", 
                (width // 2 - 200, 23), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)

# Show the final result
plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB))
plt.title("Surveillance Analytics Output")
plt.axis('off')
plt.show()

## 6. Video File Processing and Analytics

This section demonstrates how to process a surveillance video file frame-by-frame, applying the same YOLOv11 tracking and restricted zone intrusion detection.

We download a sample surveillance video of people walking, run inference on each frame, overlay bounding boxes and restricted zones, save the output as an MP4 video, convert it to an H.264 web-compatible format, and play it directly inside Google Colab.

In [ ]:
import urllib.request

# Download a sample surveillance video of people walking
video_url = "https://github.com/intel-iot-devkit/sample-videos/raw/master/face-demographics-walking-and-pause.mp4"
video_input_path = "surveillance_input.mp4"
video_output_path = "surveillance_output_raw.mp4"
video_final_path = "surveillance_output.mp4"

if not os.path.exists(video_input_path):
    print("Downloading sample video...")
    urllib.request.urlretrieve(video_url, video_input_path)
    print("Download completed!")
else:
    print("Sample video already exists.")

In [ ]:
# Open the input video
cap = cv2.VideoCapture(video_input_path)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"Video Resolution: {width}x{height}, FPS: {fps}, Total Frames: {total_frames}")

# Scale the restricted zone polygon to the video's actual resolution
video_scaled_points = []
for pt in normalized_polygon:
    video_scaled_points.append([int(pt[0] * width), int(pt[1] * height)])
video_scaled_polygon = np.array(video_scaled_points, dtype=np.int32).reshape((-1, 1, 2))

# Setup VideoWriter for raw output
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(video_output_path, fourcc, fps, (width, height))

print("Processing video frames (this may take a minute)...")
frame_idx = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
        
    frame_idx += 1
    if frame_idx % 30 == 0 or frame_idx == 1:
        print(f"Processing frame {frame_idx}/{total_frames}...")
        
    # Run YOLO tracking inference on the frame (classes=[0] tracks persons only)
    results = model.track(source=frame, persist=True, conf=0.25, classes=[0], imgsz=320, verbose=False)
    
    predictions = []
    zone_intruded = False
    annotated_frame = frame.copy()
    
    if results and len(results) > 0:
        result = results[0]
        boxes = result.boxes
        
        for i in range(len(boxes)):
            box = boxes[i]
            track_id = int(box.id[0].item()) if box.id is not None else 1
            class_id = int(box.cls[0].item())
            class_name = model.names[class_id]
            confidence = float(box.conf[0].item())
            
            xyxy = box.xyxy[0].cpu().numpy()
            x1, y1, x2, y2 = map(int, xyxy)
            
            # Check intrusion (point in polygon test on box center)
            center_x = (x1 + x2) // 2
            center_y = (y1 + y2) // 2
            point = (float(center_x), float(center_y))
            
            is_inside = cv2.pointPolygonTest(video_scaled_polygon, point, measureDist=False) >= 0
            if is_inside:
                zone_intruded = True
                
            predictions.append({
                "track_id": track_id,
                "class_name": class_name,
                "confidence": confidence,
                "box": [x1, y1, x2, y2],
                "intruded": is_inside
            })
            
    # Draw overlays
    overlay = annotated_frame.copy()
    zone_color = (0, 0, 255) if zone_intruded else (255, 255, 0) # Red if intruded, Cyan if clear
    cv2.fillPoly(overlay, [video_scaled_polygon], zone_color)
    cv2.addWeighted(overlay, 0.15, annotated_frame, 0.85, 0, annotated_frame)
    cv2.polylines(annotated_frame, [video_scaled_polygon], isClosed=True, color=zone_color, thickness=2)
    
    for pred in predictions:
        box = pred["box"]
        x1, y1, x2, y2 = box
        intruded = pred["intruded"]
        box_color = (0, 0, 255) if intruded else (0, 255, 0)
        
        # Bounding box
        cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), box_color, 2)
        
        # Label
        label = f"{pred['class_name'].upper()} #{pred['track_id']} {int(pred['confidence']*100)}%"
        (text_w, text_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.4, 1)
        cv2.rectangle(annotated_frame, (x1, y1 - text_h - 6), (x1 + text_w + 10, y1), box_color, -1)
        cv2.putText(annotated_frame, label, (x1 + 5, y1 - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255) if intruded else (0, 0, 0), 1)
        
        # Center dot
        center_x = (x1 + x2) // 2
        center_y = (y1 + y2) // 2
        cv2.circle(annotated_frame, (center_x, center_y), 4, box_color, -1)
        
    # Warning intrusion banner
    if zone_intruded:
        cv2.rectangle(annotated_frame, (0, 0), (width, 35), (0, 0, 255), -1)
        cv2.putText(annotated_frame, "!!! INTRUSION DETECTED IN RESTRICTED ZONE !!!", 
                    (width // 2 - 200, 23), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
                    
    # Write the annotated frame to output video
    out.write(annotated_frame)

cap.release()
out.release()
print("Video processing finished and saved to raw output.")

In [ ]:
# Convert the output video to H.264 using ffmpeg for HTML5 browser compatibility in Google Colab
print("Converting processed video to H.264 for inline notebook playback...")
os.system(f"ffmpeg -y -i {video_output_path} -vcodec libx264 -f mp4 {video_final_path}")
print("Conversion complete!")

In [ ]:
# Display the processed video in the Google Colab notebook
from IPython.display import HTML
from base64 import b64encode

print("Loading video player...")
mp4 = open(video_final_path, 'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

HTML(f"""
<div style="text-align: center;">
    <h3 style="color: #2c3e50; font-family: sans-serif;">Processed Surveillance Stream</h3>
    <video width="640" height="480" controls style="border: 2px solid #34495e; border-radius: 8px; box-shadow: 0 4px 8px rgba(0,0,0,0.15);">
        <source src="{data_url}" type="video/mp4">
        Your browser does not support the video tag.
    </video>
</div>
""")

## 7. Real-Time Live Webcam Stream Detection (Google Colab)

Since Google Colab runs on a remote cloud server, it cannot directly access your local webcam hardware using standard OpenCV functions like `cv2.VideoCapture(0)`.

To resolve this, we build a **JavaScript + Python Bridge** inside the browser:
1. **JavaScript** requests webcam access, plays the camera feed in a local `<video>` element, and overlays a transparent canvas on top.
2. In a continuous loop, JavaScript grabs frames from the camera, base64-encodes them, and passes them to Python.
3. **Python** decodes each base64 frame into an OpenCV BGR image, scales the restricted zone polygon, runs the YOLOv11 tracker, checks for intrusions, and draws overlays on a transparent RGBA image.
4. Python base64-encodes the overlay image and returns it to JavaScript, which instantly updates the transparent canvas overlay on the webcam video.

In [ ]:
# Import Colab-specific tools and base64 encoders
from IPython.display import display, Javascript, HTML
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import io
import PIL.Image

# Helper function to convert JavaScript base64 reply to OpenCV BGR image
def js_to_image(js_reply):
    """
    Converts a base64 encoded JavaScript string into an OpenCV image.
    """
    image_bytes = b64decode(js_reply.split(',')[1])
    jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8)
    img = cv2.imdecode(jpg_as_np, flags=1)
    return img

# Helper function to convert transparent OpenCV overlay to base64 bytes for JS display
def bbox_to_bytes(bbox_array):
    """
    Converts a numpy transparent overlay (RGBA) to a base64 PNG byte string.
    """
    bbox_PIL = PIL.Image.fromarray(bbox_array, 'RGBA')
    iobuf = io.BytesIO()
    bbox_PIL.save(iobuf, format='png')
    bbox_bytes = 'data:image/png;base64,{}'.format((str(b64encode(iobuf.getvalue()), 'utf-8')))
    return bbox_bytes

In [ ]:
# Define the JavaScript webcam stream controller
# This sets up the HTML video player, overlay canvas, and handles frame extraction and rendering.

def start_webcam_stream():
    js_code = """
    var video;
    var container;
    var canvas;
    var labelVal;
    
    async function initWebcam() {
      // Create modern styling container for the webcam feed
      container = document.createElement('div');
      container.style.display = 'flex';
      container.style.flexDirection = 'column';
      container.style.alignItems = 'center';
      container.style.fontFamily = 'system-ui, -apple-system, sans-serif';
      container.style.background = '#1e1e24';
      container.style.padding = '20px';
      container.style.borderRadius = '16px';
      container.style.boxShadow = '0 10px 25px rgba(0,0,0,0.3)';
      container.style.width = '680px';
      container.style.margin = '20px auto';
      container.style.color = '#ffffff';
      
      // Title
      const title = document.createElement('h3');
      title.innerText = 'Live Restricted Zone Surveillance Stream';
      title.style.margin = '0 0 15px 0';
      title.style.color = '#3b82f6';
      title.style.fontSize = '1.3rem';
      container.appendChild(title);
      
      // Wrapper for video + overlay canvas positioning
      const videoWrapper = document.createElement('div');
      videoWrapper.style.position = 'relative';
      videoWrapper.style.width = '640px';
      videoWrapper.style.height = '480px';
      videoWrapper.style.borderRadius = '8px';
      videoWrapper.style.overflow = 'hidden';
      videoWrapper.style.border = '2px solid #2d3748';
      
      video = document.createElement('video');
      video.style.display = 'block';
      video.width = 640;
      video.height = 480;
      videoWrapper.appendChild(video);
      
      canvas = document.createElement('canvas');
      canvas.style.position = 'absolute';
      canvas.style.top = '0';
      canvas.style.left = '0';
      canvas.width = 640;
      canvas.height = 480;
      videoWrapper.appendChild(canvas);
      container.appendChild(videoWrapper);
      
      // Status message display
      labelVal = document.createElement('div');
      labelVal.innerText = 'Initializing Webcam Stream...';
      labelVal.style.marginTop = '15px';
      labelVal.style.fontSize = '0.95rem';
      labelVal.style.fontWeight = 'bold';
      labelVal.style.color = '#a0aec0';
      container.appendChild(labelVal);
      
      // Stop Button
      const stopBtn = document.createElement('button');
      stopBtn.innerText = 'STOP SURVEILLANCE';
      stopBtn.style.marginTop = '15px';
      stopBtn.style.padding = '10px 20px';
      stopBtn.style.background = '#ef4444';
      stopBtn.style.color = 'white';
      stopBtn.style.border = 'none';
      stopBtn.style.borderRadius = '6px';
      stopBtn.style.cursor = 'pointer';
      stopBtn.style.fontWeight = 'bold';
      stopBtn.style.letterSpacing = '0.05em';
      stopBtn.style.transition = 'background 0.2s';
      stopBtn.onmouseover = () => stopBtn.style.background = '#dc2626';
      stopBtn.onmouseout = () => stopBtn.style.background = '#ef4444';
      
      stopBtn.onclick = () => {
        window.webcamActive = false;
        labelVal.innerText = 'Surveillance Stopped Successfully.';
        labelVal.style.color = '#ef4444';
        const tracks = video.srcObject.getTracks();
        tracks.forEach(track => track.stop());
        video.srcObject = null;
        stopBtn.disabled = true;
        stopBtn.style.background = '#4a5568';
        stopBtn.style.cursor = 'not-allowed';
      };
      container.appendChild(stopBtn);
      
      document.body.appendChild(container);
      
      // Start user webcam
      const stream = await navigator.mediaDevices.getUserMedia({ video: { width: 640, height: 480 } });
      video.srcObject = stream;
      await video.play();
      
      window.webcamActive = true;
      labelVal.innerText = 'Surveillance Active - Restricted Zone Monitored';
      labelVal.style.color = '#10b981';
    }
    
    // Capture a single frame from the video stream
    async function captureFrame() {
      if (!window.webcamActive) return null;
      
      const tempCanvas = document.createElement('canvas');
      tempCanvas.width = 640;
      tempCanvas.height = 480;
      tempCanvas.getContext('2d').drawImage(video, 0, 0, 640, 480);
      
      return tempCanvas.toDataURL('image/jpeg', 0.8);
    }
    
    // Render transparent overlay PNG on top of video element
    function drawOverlay(base64Png, isIntruded) {
      if (!window.webcamActive) return;
      const ctx = canvas.getContext('2d');
      ctx.clearRect(0, 0, 640, 480);
      
      if (base64Png) {
        const img = new Image();
        img.onload = function() {
          ctx.drawImage(img, 0, 0);
        };
        img.src = base64Png;
      }
      
      if (isIntruded) {
        labelVal.innerText = '!!! INTRUSION DETECTED IN RESTRICTED ZONE !!!';
        labelVal.style.color = '#ef4444';
        container.style.border = '2px solid #ef4444';
      } else {
        labelVal.innerText = 'Surveillance Active - Restricted Zone Monitored';
        labelVal.style.color = '#10b981';
        container.style.border = 'none';
      }
    }
    
    // Cleanup elements when stream is stopped
    function cleanupWebcam() {
      if (container && container.parentNode) {
        container.parentNode.removeChild(container);
      }
    }
    """
    display(Javascript(js_code))

In [ ]:
# Run the interactive real-time webcam surveillance loop

# Initialize webcam element in frontend
start_webcam_stream()
eval_js('initWebcam()')

print("Webcam initialized. Surveillance stream is starting... Press 'STOP SURVEILLANCE' inside the stream widget to end.")

# We will use the same scaled polygon coordinates as defined in Section 3
# since our webcam stream resolution is 640x480.
height, width = 480, 640

# Scaling points to actual pixels (640x480)
webcam_scaled_points = []
for pt in normalized_polygon:
    webcam_scaled_points.append([int(pt[0] * width), int(pt[1] * height)])
webcam_scaled_polygon = np.array(webcam_scaled_points, dtype=np.int32).reshape((-1, 1, 2))

# Real-time loop
try:
    while True:
        # Get frame from JavaScript webcam
        js_reply = eval_js('captureFrame()')
        if js_reply is None:
            break
            
        # Convert base64 frame to OpenCV BGR image
        frame = js_to_image(js_reply)
        
        # Run YOLOv11 tracker (classes=[0] tracks persons only)
        results = model.track(source=frame, persist=True, conf=0.25, classes=[0], imgsz=320, verbose=False)
        
        # Create transparent overlay (RGBA - alpha initialized to 0)
        overlay_array = np.zeros((480, 640, 4), dtype=np.uint8)
        zone_intruded = False
        predictions = []
        
        if results and len(results) > 0:
            result = results[0]
            boxes = result.boxes
            
            for i in range(len(boxes)):
                box = boxes[i]
                track_id = int(box.id[0].item()) if box.id is not None else 1
                class_id = int(box.cls[0].item())
                class_name = model.names[class_id]
                confidence = float(box.conf[0].item())
                
                xyxy = box.xyxy[0].cpu().numpy()
                x1, y1, x2, y2 = map(int, xyxy)
                
                # Check intrusion (point in polygon test on box center)
                center_x = (x1 + x2) // 2
                center_y = (y1 + y2) // 2
                point = (float(center_x), float(center_y))
                
                is_inside = cv2.pointPolygonTest(webcam_scaled_polygon, point, measureDist=False) >= 0
                if is_inside:
                    zone_intruded = True
                    
                predictions.append({
                    "track_id": track_id,
                    "class_name": class_name,
                    "confidence": confidence,
                    "box": [x1, y1, x2, y2],
                    "intruded": is_inside
                })
        
        # Draw the zone polygon on the transparent overlay (with a transparent fill)
        zone_color_rgba = (0, 0, 255, 38) if zone_intruded else (255, 255, 0, 38) # Red if intruded, Cyan if clear
        zone_border_rgba = (0, 0, 255, 255) if zone_intruded else (255, 255, 0, 255)
        
        # Draw transparent filled polygon
        cv2.fillPoly(overlay_array, [webcam_scaled_polygon], zone_color_rgba)
        cv2.polylines(overlay_array, [webcam_scaled_polygon], isClosed=True, color=zone_border_rgba, thickness=2)
        
        # Draw detections on the transparent overlay
        for pred in predictions:
            box = pred["box"]
            x1, y1, x2, y2 = box
            intruded = pred["intruded"]
            
            # Colors in RGBA
            box_color_rgba = (0, 0, 255, 255) if intruded else (0, 255, 0, 255) # Red for intruder, Green for safe
            
            # Draw Bounding Box
            cv2.rectangle(overlay_array, (x1, y1), (x2, y2), box_color_rgba, 2)
            
            # Draw Label Background and Text
            label = f"{pred['class_name'].upper()} #{pred['track_id']} {int(pred['confidence']*100)}%"
            (text_w, text_h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.4, 1)
            cv2.rectangle(overlay_array, (x1, y1 - text_h - 6), (x1 + text_w + 10, y1), box_color_rgba, -1)
            
            text_color = (255, 255, 255, 255) if intruded else (0, 0, 0, 255)
            cv2.putText(overlay_array, label, (x1 + 5, y1 - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.4, text_color, 1)
            
            # Draw center point
            center_x = (x1 + x2) // 2
            center_y = (y1 + y2) // 2
            cv2.circle(overlay_array, (center_x, center_y), 4, box_color_rgba, -1)
            
        # Draw Warning Header Banner on the overlay
        if zone_intruded:
            cv2.rectangle(overlay_array, (0, 0), (640, 35), (0, 0, 255, 255), -1)
            cv2.putText(overlay_array, "!!! INTRUSION DETECTED IN RESTRICTED ZONE !!!", 
                        (120, 23), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255, 255), 2)
                        
        # Convert overlay to base64 bytes for drawing in JS
        bbox_bytes = bbox_to_bytes(overlay_array)
        
        # Send transparent overlay back to JS to render
        eval_js(f'drawOverlay("{bbox_bytes}", {str(zone_intruded).lower()})')

except Exception as e:
    print(f"Stream stopped or closed: {e}")